<h2>Utilities</h2>

In [82]:
import docx
from PyPDF2 import PdfReader

In [83]:
import os
from dotenv import load_dotenv
load_dotenv(override=True) 

AI_KEY = os.environ['AI_KEY']
AI_MODEL_NAME = os.environ['AI_MODEL_NAME']

from google import genai

client = genai.Client(api_key=AI_KEY)

python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 9
python-dotenv could not parse statement starting at line 11
python-dotenv could not parse statement starting at line 15
python-dotenv could not parse statement starting at line 16
python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 18
python-dotenv could not parse statement starting at line 19
python-dotenv could not parse statement starting at line 20
python-dotenv could not parse statement starting at line 21
python-dotenv could not parse statement starting at line 23
python-dotenv could not parse statement starting at line 26


In [84]:
AI_JSON_RESPONSE_TYPE = {"response_mime_type": "application/json"}
AI_TEXT_RESPONSE_TYPE = {"response_mime_type": "text/plain"}

def call_ai(generation_config, prompt):
    try:
        response = client.models.generate_content(
            model=AI_MODEL_NAME,
            config=generation_config,
            contents=prompt
        )
        return response.text
    except Exception as e:
        print('query_ai Exception', e)
        return ''

In [85]:
def call_ai_for_text(prompt):
    return call_ai(AI_TEXT_RESPONSE_TYPE, prompt)

In [86]:
TEST_PDF_FILEPATH = 'samples/cv1.pdf'


In [87]:

def extract_text_from_file(file_path):
    if file_path.endswith(".docx"):
        doc = docx.Document(file_path)
        return "\n".join([p.text for p in doc.paragraphs])

    elif file_path.endswith(".pdf"):
        reader = PdfReader(file_path)
        return "".join([page.extract_text() or "" for page in reader.pages])

    else:
        raise ValueError("Unsupported file format")

#
raw_cv_content = extract_text_from_file(TEST_PDF_FILEPATH)

<h2>Agent 1 — Evaluator</h2>

In [88]:
def evaluator_agent(cv_text, job_text):
    prompt = f"""
You are an ATS evaluator.

Return JSON:
{{
  "score": 0-100,
  "keyword_match": 0-100,
  "experience_match": 0-100,
  "skills_missing": [],
  "critical_missing": [],
  "summary": ""
}}

Job:
{job_text}

CV:
{cv_text}
"""
    return call_ai_for_text(prompt)

<h2>Agent 2 — Gatekeeper</h2>
Decide user should apply this job or not

In [89]:
def gatekeeper_agent(eval_result):
    prompt = f"""
Decide whether to optimize CV.

Rules:
- If critical skills missing → REJECT
- If experience strong but keyword weak → OPTIMIZE
- If too far from job → REJECT

Return:
{{
  "decision": "REJECT | OPTIMIZE",
  "reason": "",
  "confidence": 0-100
}}

Evaluation:
{eval_result}
"""
    return call_ai(AI_JSON_RESPONSE_TYPE, prompt)

<h2>Agent 3 — Optimizer</h2>

In [90]:
def optimizer_agent(cv_text, job_text):
    prompt = f"""
Rewrite CV to improve ATS score.

Return structured JSON:
{{
  "summary": "",
  "skills": [],
  "experience": []
}}

Rules:
- Add missing skills if reasonable
- Do NOT fabricate experience
- Improve bullet points with metrics

Job:
{job_text}

CV:
{cv_text}
"""
    return call_ai_for_text(prompt)

<h2>Agent 4 — Validator</h2>
Detect fake improvements

In [91]:
def validator_agent(old_eval, new_eval):
    prompt = f"""
Check if CV improvement is REAL.

Rules:
- If score improves but skills still missing → FAKE
- If improvement < 10 → NOT USEFUL

Return:
{{
  "is_valid": true/false,
  "improvement_score": number,
  "issues": []
}}

Before:
{old_eval}

After:
{new_eval}
"""
    return call_ai_for_text(prompt)

<h2>Agent 5 — Final Decision</h2>
Give results for user to decide applying or not

In [92]:
def decision_agent(old_eval, new_eval, validation):
    prompt = f"""
Make final decision:

Options:
- REJECT
- APPLY_WITH_CAUTION
- APPLY

Return:
{{
  "final_decision": "",
  "confidence": 0-100,
  "reason": "",
  "should_apply": true/false
}}

Before:
{old_eval}

After:
{new_eval}

Validation:
{validation}
"""
    return call_ai_for_text(prompt)

<h2>Agent 6 — Formatter</h2>
Generate new CV

<h2>Orchestrator (Main Brain)</h2>

In [93]:
TEST_JOB_DETAILS = """
Full job description
Job description
Ahamove is looking for a Junior Backend Software Engineer to join our engineering team and help build scalable systems that process hundreds of thousands of orders every day. You will work in a fast-moving environment with challenging problems in performance, reliability, distributed systems, and real-time data processing.
This role is ideal for those who are eager to learn, grow quickly, and work with large-scale backend architectures and cloud infrastructure.
Key Responsibilities
Develop, maintain, and optimize backend services (microservices).
Build and integrate APIs for customers, drivers, partners, and internal systems.
Participate in designing system architecture, data models, and processing flows.
Write clean, maintainable, well-structured code following clean architecture principles.
Troubleshoot issues, fix bugs, and improve service performance.
Collaborate closely with Product Managers, QA, Data teams, and other engineers.
Participate in code reviews and continuously improve team coding standards.
Job requirement
1-2 years of experience as a Backend Engineer.
Bachelor's degree in Computer Science, Software Engineering, or related fields.
Proficiency in Python or Golang (at least one required).
Solid understanding of backend fundamentals: REST APIs, databases, HTTP, data structures, algorithms.
Basic SQL knowledge and experience with MongoDB or PostgreSQL
Strong problem-solving skills, growth mindset, and high ownership.
Nice-to-have
Experience with Node.js or JavaScript.
Understanding of microservices architecture.
Experience working with Kubernetes (K8s) or similar container orchestration systems.
Experience with AWS or other cloud platforms.
Familiarity with Docker, CI/CD, or message queues (Kafka, RabbitMQ).
Exposure to clean architecture or domain-driven design (DDD).
Benefit
Physical Wellbeing Benefit: General Insurance, Medical check-up, Accident Insurance, Healthcare Insurance
Emotional Wellbeing Benefit: Company Trip, Year End Party, Aha Hour Activities, Special Day Gifts, Aha Club (Badminton, Soccer)
Financial Wellbeing Benefit: Grab/Be For Work, Workplace Relocation, 13th Month Salary, PP Appreciate, Annual Leave Remain, ESOP
"""

In [94]:
def run_pipeline(cv_text, job_text):
    # 1. Evaluate
    eval1 = evaluator_agent(cv_text, job_text)

    # 2. Gatekeeper
    gate = gatekeeper_agent(eval1)

    if gate["decision"] == "REJECT":
        return {
            "status": "REJECTED",
            "reason": gate
        }

    # 3. Optimize
    new_cv = optimizer_agent(cv_text, job_text)

    # 4. Convert to text
    new_text = str(new_cv)

    # 5. Re-evaluate
    eval2 = evaluator_agent(new_text, job_text)

    # 6. Validate
    validation = validator_agent(eval1, eval2)

    if not validation["is_valid"]:
        return {
            "status": "REJECTED_AFTER_OPTIMIZATION",
            "validation": validation
        }

    # 7. Final decision
    final = decision_agent(eval1, eval2, validation)

    return {
        "status": "DONE",
        "before": eval1,
        "after": eval2,
        "final": final,
        "cv": new_cv
    }

#test
run_pipeline(raw_cv_content, TEST_JOB_DETAILS)

query_ai Exception 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


TypeError: string indices must be integers, not 'str'